<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Builds `diploma_type_bucket` — a train-driven categorical feature that groups
raw `diploma_type_id` codes into top-5 individual buckets, a rare-codes bucket (6), and an
unseen/null bucket (-1).  
Does NOT process `diploma_gpa` (added directly to MODEL_FEATURES in a later step).

**Prerequisite:** `merge_diploma.py` must have been run BEFORE `split_diagnostics.ipynb`
so that `diploma_type_id` and `diploma_gpa` exist in the split parquet files on disk.

**Notebook Shape:** Steps A (Load) + B (Bucketing engineering) + Save.

**Inputs / Data Sources:**
- `df_train = pd.read_parquet(TRAIN_PATH)` — bucketing rules learned from this split ONLY
- `df_valid = pd.read_parquet(VALID_PATH)`
- `df_test  = pd.read_parquet(TEST_PATH)`

**Outputs / Side Effects:**
- `df_train.to_parquet(TRAIN_PATH, index=True)` — adds `diploma_type_bucket`
- `df_valid.to_parquet(VALID_PATH, index=True)` — adds `diploma_type_bucket`
- `df_test.to_parquet(TEST_PATH,   index=True)` — adds `diploma_type_bucket`

**Logic Flow:**
1. Load splits from `SPLIT_DATA_DIR`; assert `diploma_type_id` is present.
2. Compute `diploma_type_id.value_counts()` from `df_train` ONLY.
3. Identify top-5 codes and apply bucket mapping (raw codes kept for top-5; rare → 6; unseen → -1).
4. Assert exactly ONE new column was added before saving.

**Maintainability Notes:**
Do NOT update `MODEL_FEATURES`, `CATEGORICAL_FEATURES`, or `feature_contract.json` here.
Review the printed diagnostics first; update model training as a separate follow-up step.

# Diploma Type Bucketing

**Step A** — Load `df_train`, `df_valid`, `df_test` from the pre-built parquet files saved
by `split_diagnostics.ipynb` (after `merge_diploma.py` added `diploma_type_id`).  
**Step B** — Build `diploma_type_bucket` from train-only aggregation and apply to all splits.  
**Do NOT modify** `MODEL_FEATURES`, `CATEGORICAL_FEATURES`, or `feature_contract.json` here.

## Step A — Load Pre-built Splits

The temporal split was defined and saved by `split_diagnostics.ipynb`.
`merge_diploma.py` must have been run (and `split_diagnostics.ipynb` rerun after it)
before loading here, so that `diploma_type_id` exists in the parquet files.
This notebook **never re-derives splits** — it only reads them.

**Step A.0 — Configuration.** `SPLIT_DATA_DIR` must match the same constant in
`split_diagnostics.ipynb` and `course_difficulty.ipynb`. Edit this path only if
the split files were moved — never change the split logic here.

In [1]:
SPLIT_DATA_DIR = 'D:/AI/Real projects/Academic_Advisor/data/model_data'

**Step A.1** — Load `df_train`, `df_valid`, `df_test` from the saved parquet files.
Print each shape immediately to confirm the files exist and have the expected row counts.

In [2]:
import pandas as pd
import numpy as np
import os

TRAIN_PATH = os.path.join(SPLIT_DATA_DIR, 'df_train.parquet')
VALID_PATH = os.path.join(SPLIT_DATA_DIR, 'df_valid.parquet')
TEST_PATH  = os.path.join(SPLIT_DATA_DIR, 'df_test.parquet')

df_train = pd.read_parquet(TRAIN_PATH)
df_valid = pd.read_parquet(VALID_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'df_train : {df_train.shape}')
print(f'df_valid : {df_valid.shape}')
print(f'df_test  : {df_test.shape}')

df_train : (450465, 70)
df_valid : (156097, 70)
df_test  : (110008, 70)


**Step A.1b — Remove stale `diploma_type_bucket` columns (idempotent clean-up).**
If this notebook has been run before, the parquet files on disk already carry
`diploma_type_bucket`. Drop it now so the column count returns to the base and the
assertion in Step B.10 stays accurate. This cell is safe to run multiple times.

In [ ]:
# ── Step A.1b: Drop stale diploma_type_bucket if present ──

_STALE_COLS = ['diploma_type    _bucket']

for _df in [df_train, df_valid, df_test]:
    _to_drop = [c for c in _STALE_COLS if c in _df.columns]
    if _to_drop:
        _df.drop(columns=_to_drop, inplace=True)

print('Base column count after removing stale bucket columns:')
print(f'  df_train : {df_train.shape[1]}')
print(f'  df_valid : {df_valid.shape[1]}')
print(f'  df_test  : {df_test.shape[1]}')
print('(All three must match — any mismatch means the clean-up failed.)')

assert df_train.shape[1] == df_valid.shape[1] == df_test.shape[1], (
    'FAIL: Base column counts differ across splits after dropping stale columns.'
)

# ── Snapshot base column counts (used later in Step B.7 to verify
# exactly one new column was added — captured HERE, right after stale
# columns are dropped, so it's an independent measurement, not derived
# backward from the post-bucketing shape) ──
BASE_COL_COUNT_TRAIN = df_train.shape[1]
BASE_COL_COUNT_VALID = df_valid.shape[1]
BASE_COL_COUNT_TEST  = df_test.shape[1]

print()
print('Captured base column counts (pre-bucketing snapshot):')
print(f'  BASE_COL_COUNT_TRAIN : {BASE_COL_COUNT_TRAIN}')
print(f'  BASE_COL_COUNT_VALID : {BASE_COL_COUNT_VALID}')
print(f'  BASE_COL_COUNT_TEST  : {BASE_COL_COUNT_TEST}')

Base column count after removing stale bucket columns:
  df_train : 70
  df_valid : 70
  df_test  : 70
(All three must match — any mismatch means the clean-up failed.)

Captured base column counts (pre-bucketing snapshot):
  BASE_COL_COUNT_TRAIN : 70
  BASE_COL_COUNT_VALID : 70
  BASE_COL_COUNT_TEST  : 70


**Step A.1c — Assert `diploma_type_id` EXISTS in all three splits (existence only — nulls are ALLOWED).**

`diploma_type_id` is added to `after_fet_eng.parquet` by `merge_diploma.py`,
then propagated to the splits when `split_diagnostics.ipynb` is rerun.
If the column is missing here, the split files are stale.

**Null policy (locked):** missing/unmatched `diploma_type_id` is an EXPECTED,
allowed case (students with no diploma record in the source). This cell does
NOT assert non-null — it only asserts the column exists, then prints null
counts loudly so the scale of missingness is visible before bucketing. Null
rows are mapped to the unseen bucket (`-1`) in Step B.4/B.5, same as any
code never seen in train.

**DO NOT fix a missing column by re-joining from the diploma source table here.**
A re-join risks duplicating rows, reordering the index, or introducing a different
dtype than what the merge script produced. Instead:
1. Re-run `merge_diploma.py` to update `after_fet_eng.parquet`.
2. Then re-run `split_diagnostics.ipynb` to rebuild the split parquet files.
3. Then reload here.

In [4]:
# ── Step A.1c: Assert diploma_type_id exists in all three splits ──

for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    if 'diploma_type_id' not in _df_split.columns:
        raise AssertionError(
            f'\nFAIL: diploma_type_id is missing from the {_split_name} split.\n'
            f'The parquet files were built before merge_diploma.py was run.\n'
            f'Fix:\n'
            f'  1. Run merge_diploma.py to add diploma_type_id to after_fet_eng.parquet.\n'
            f'  2. Re-run split_diagnostics.ipynb to rebuild the split parquet files.\n'
            f'  3. Reload this notebook.\n'
            f'DO NOT fix this by merging diploma_type_id here — that risks row reordering.'
        )

print('diploma_type_id present in all three splits: OK  (existence check only — nulls are allowed and expected)')
print()
print('diploma_type_id null percentage per split:')
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n_total = len(_df_split)
    _n_null  = int(_df_split['diploma_type_id'].isna().sum())
    _pct     = _n_null / _n_total * 100
    print(f'  {_split_name:<6}: {_n_null:>7,} / {_n_total:>7,} null  ({_pct:.4f}%)')
print()
print('diploma_gpa null percentage per split:')
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n_total = len(_df_split)
    _n_null  = int(_df_split['diploma_gpa'].isna().sum()) if 'diploma_gpa' in _df_split.columns else _n_total
    _pct     = _n_null / _n_total * 100
    print(f'  {_split_name:<6}: {_n_null:>7,} / {_n_total:>7,} null  ({_pct:.4f}%)')

diploma_type_id present in all three splits: OK  (existence check only — nulls are allowed and expected)

diploma_type_id null percentage per split:
  train :      31 / 450,465 null  (0.0069%)
  valid :       0 / 156,097 null  (0.0000%)
  test  :      17 / 110,008 null  (0.0155%)

diploma_gpa null percentage per split:
  train :      31 / 450,465 null  (0.0069%)
  valid :       0 / 156,097 null  (0.0000%)
  test  :      17 / 110,008 null  (0.0155%)


**What this shows:** Presence and null coverage of `diploma_type_id` and `diploma_gpa`.
Null rows come from students who had no diploma record in the source — expected for
some student populations. The bucketing step will map those nulls to `-1` (unseen bucket).
If the assertion fires, the fix is always to re-run the two upstream scripts — never to
merge inside this notebook.

**Step A.2** — Confirm the loaded splits have the expected year boundaries
(train 2005–2021, val 2022–2023, test 2024–2025s1).  
If boundaries are wrong, stop and re-run `split_diagnostics.ipynb` first.

In [5]:
_rows = []
for _name, _df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n = len(_df)
    _rows.append({
        'split':     _name,
        'row_count': _n,
        'min_year':  _df['part_year'].min(),
        'max_year':  _df['part_year'].max(),
    })
print(pd.DataFrame(_rows).to_string(index=False))

split  row_count  min_year  max_year
train     450465      2005      2021
valid     156097      2022      2023
 test     110008      2024      2025


**What this shows:** Min/max year per split confirms the correct temporal boundaries.
Train must be 2005–2021, valid 2022–2023, test 2024–2025.
Any deviation means the split files were built with different logic —
stop and re-run `split_diagnostics.ipynb`.

---

## Step B — Diploma Type Bucketing (train-only aggregation)

All aggregations in this step use **`df_train` exclusively**.
`df_valid` and `df_test` only receive the mapping — they never enter any aggregate.
This mirrors the same principle used for course difficulty in `course_difficulty.ipynb`.

**Step B.1** — Compute `diploma_type_id.value_counts()` from `df_train` ONLY.
Print the full distribution so the code-frequency landscape is visible before any
bucketing decision is made.

In [6]:
# All aggregations are from df_train ONLY — df_valid and df_test never touched here.

_train_vc = (
    df_train['diploma_type_id']
    .value_counts(dropna=False)
    .sort_values(ascending=False)
)

print(f'diploma_type_id distribution in df_train (all codes, sorted by count):')
print(f'  Total rows in train : {len(df_train):,}')
print(f'  Distinct codes      : {df_train["diploma_type_id"].nunique(dropna=True)}')
print(f'  Null rows           : {int(df_train["diploma_type_id"].isna().sum()):,}')
print()
print(_train_vc.rename('count').to_frame().to_string())

diploma_type_id distribution in df_train (all codes, sorted by count):
  Total rows in train : 450,465
  Distinct codes      : 14
  Null rows           : 31

                  count
diploma_type_id        
15.0             426684
16.0              13701
13.0               5598
19.0               2334
26.0                881
21.0                322
32.0                274
10.0                167
18.0                131
22.0                101
20.0                 83
14.0                 72
9.0                  43
69.0                 43
NaN                  31


**What this shows:** The full frequency distribution of `diploma_type_id` codes in the
training set. Large counts indicate common diploma types; very small counts are candidates
for the rare bucket. Use this table to sanity-check that the top-5 selection in Step B.2
captures the majority of the training population.

**Step B.2** — Select the top-5 codes by count (train-only). Print them explicitly with
their counts. These codes will be kept as their raw integer values in `diploma_type_bucket`
— no relabelling to ranks 1–5 (same convention as `requirement_type_id` in model_training.py).

In [7]:
# Top-5 by count, excluding null rows from the ranking.
_train_vc_notnull = (
    df_train['diploma_type_id']
    .dropna()
    .value_counts()
    .sort_values(ascending=False)
)

TOP_DIPLOMA_CODES = list(_train_vc_notnull.index[:5])
_top5_set         = set(TOP_DIPLOMA_CODES)

print('TOP_DIPLOMA_CODES (top 5 by count in df_train, raw integer values):')
for _rank, _code in enumerate(TOP_DIPLOMA_CODES, start=1):
    _cnt = int(_train_vc_notnull[_code])
    _pct = _cnt / len(df_train) * 100
    print(f'  Rank {_rank}: code={int(_code)}   count={_cnt:>8,}  ({_pct:.2f}% of train)')

print(f'\nTop-5 codes cover '
      f'{_train_vc_notnull.head(5).sum():,} / '
      f'{int(df_train["diploma_type_id"].notna().sum()):,} '
      f'non-null train rows '
      f'({_train_vc_notnull.head(5).sum() / int(df_train["diploma_type_id"].notna().sum()) * 100:.2f}%)')

TOP_DIPLOMA_CODES (top 5 by count in df_train, raw integer values):
  Rank 1: code=15   count= 426,684  (94.72% of train)
  Rank 2: code=16   count=  13,701  (3.04% of train)
  Rank 3: code=13   count=   5,598  (1.24% of train)
  Rank 4: code=19   count=   2,334  (0.52% of train)
  Rank 5: code=26   count=     881  (0.20% of train)

Top-5 codes cover 449,198 / 450,434 non-null train rows (99.73%)


**What this shows:** The 5 most frequent diploma type codes and the fraction of the
training population they represent. If the top 5 cover < 80% of non-null train rows,
consider whether 5 is the right cut-off. Change `TOP_DIPLOMA_CODES` only after reviewing
the diagnostics in Step B.6.

**Step B.3 — Critical safety check: code 6 collision.**

The rare-code bucket is labelled `6`. If any real diploma type has the raw code `6`,
that code would collide with the rare-bucket label and become uninterpretable.
This check must pass before any bucketing is applied.

In [8]:
# ── Step B.3: Safety check — code 6 must not exist in any split ──
#
# The rare bucket is labelled 6. If a real diploma code of 6 exists anywhere
# in the data, it would collide with the rare-bucket label and the two would
# become indistinguishable. The rare-bucket label must then be changed to a
# value outside the real code range before proceeding.

RARE_BUCKET_LABEL   = 6
UNSEEN_BUCKET_LABEL = -1

_collision_found = False
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n_code6 = int((_df_split['diploma_type_id'] == RARE_BUCKET_LABEL).sum())
    print(f'  {_split_name:<6}: rows with diploma_type_id == {RARE_BUCKET_LABEL} : {_n_code6:,}')
    if _n_code6 > 0:
        _collision_found = True

if _collision_found:
    raise AssertionError(
        f'\nFAIL: diploma_type_id == {RARE_BUCKET_LABEL} exists in the data.\n'
        f'A real diploma code of {RARE_BUCKET_LABEL} would collide with the rare-bucket label.\n'
        f'Change RARE_BUCKET_LABEL to a value that does not appear in the real code range\n'
        f'(e.g. the maximum real code + 1, or 999) before proceeding.'
    )

print(f'\nSafety check PASSED: no real diploma_type_id == {RARE_BUCKET_LABEL} in any split.')
print(f'  Rare-bucket label  : {RARE_BUCKET_LABEL}')
print(f'  Unseen-bucket label: {UNSEEN_BUCKET_LABEL}')

  train : rows with diploma_type_id == 6 : 0
  valid : rows with diploma_type_id == 6 : 0
  test  : rows with diploma_type_id == 6 : 0

Safety check PASSED: no real diploma_type_id == 6 in any split.
  Rare-bucket label  : 6
  Unseen-bucket label: -1


**What this shows:** Confirms that the rare-bucket sentinel (`6`) does not clash with any
real diploma type code in any of the three splits. If the assertion fires, scroll back to
the `RARE_BUCKET_LABEL` definition and pick a value outside the real code range.

**Step B.4 — Build the bucketing rule.**

Mapping (all from `df_train` aggregation — `df_valid` and `df_test` never enter):
- `diploma_type_id` in `TOP_DIPLOMA_CODES` → kept as its own **raw-code** category
- `diploma_type_id` is a known train code but NOT in top 5 → mapped to `6` (rare bucket)
- `diploma_type_id` never appeared in `df_train` at all (only in valid/test) → mapped to `-1`
- `diploma_type_id` is null → mapped to `-1` (treated same as unseen)

Raw codes are kept for the top 5 — no relabelling to ranks (same convention as
`requirement_type_id` in `model_training.py::learn_categorical_levels`).

In [9]:
# ── Step B.4: Build bucketing lookup from df_train ONLY ──

# All distinct non-null codes seen in training data
_all_train_codes = set(
    int(v) for v in df_train['diploma_type_id'].dropna().unique()
)
_rare_codes      = _all_train_codes - _top5_set

print(f'All distinct diploma_type_id codes seen in df_train : {sorted(_all_train_codes)}')
print(f'Top-5 codes (own-category bucket)                   : {sorted(int(c) for c in _top5_set)}')
print(f'Rare codes  (-> bucket {RARE_BUCKET_LABEL})                           : {sorted(int(c) for c in _rare_codes)}')
print()
print(f'  {len(_top5_set)} code(s) will be kept as own raw-code categories')
print(f'  {len(_rare_codes)} code(s) will map to rare bucket ({RARE_BUCKET_LABEL})')
print(f'  Codes never seen in train will map to unseen bucket ({UNSEEN_BUCKET_LABEL})')
print(f'  Null diploma_type_id will map to unseen bucket ({UNSEEN_BUCKET_LABEL})')

# Build a lookup dict for fast per-row mapping
_bucket_map = {}
for _code in _top5_set:
    _bucket_map[int(_code)] = int(_code)   # raw code kept as-is
for _code in _rare_codes:
    _bucket_map[int(_code)] = RARE_BUCKET_LABEL

# All categories that will appear in any split
_bucket_categories = sorted([int(c) for c in _top5_set]) + [RARE_BUCKET_LABEL, UNSEEN_BUCKET_LABEL]
print(f'\nFinal pd.Categorical categories: {_bucket_categories}')


def _apply_diploma_bucket(series):
    """Map raw diploma_type_id -> bucket integer."""
    def _map_one(val):
        if pd.isna(val):
            return UNSEEN_BUCKET_LABEL  # null -> unseen
        _v = int(val)
        return _bucket_map.get(_v, UNSEEN_BUCKET_LABEL)  # not in map -> unseen
    return series.map(_map_one)

All distinct diploma_type_id codes seen in df_train : [9, 10, 13, 14, 15, 16, 18, 19, 20, 21, 22, 26, 32, 69]
Top-5 codes (own-category bucket)                   : [13, 15, 16, 19, 26]
Rare codes  (-> bucket 6)                           : [9, 10, 14, 18, 20, 21, 22, 32, 69]

  5 code(s) will be kept as own raw-code categories
  9 code(s) will map to rare bucket (6)
  Codes never seen in train will map to unseen bucket (-1)
  Null diploma_type_id will map to unseen bucket (-1)

Final pd.Categorical categories: [13, 15, 16, 19, 26, 6, -1]


**What this shows:** The complete bucket mapping built from `df_train` aggregation.
Top-5 codes are kept as raw integers; the remaining train codes go to bucket `6`;
anything not in train (including nulls) goes to `-1`.  
The `_bucket_categories` list defines the explicit `pd.Categorical` categories applied
in Step B.5 — all three splits will use this same set.

**Step B.5** — Apply the bucketing mapping to all three splits, creating
`diploma_type_bucket` as a `pd.Categorical` column with `int` codes.
The raw `diploma_type_id` column is left intact as an audit-only column
(same treatment as `difficulty_group_support_count` in `course_difficulty.ipynb`).

In [10]:
# ── Step B.5: Apply mapping to all three splits ──
#
# diploma_type_id is kept unchanged (audit-only).
# diploma_type_bucket is the new model-facing column.

for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _raw_mapped = _apply_diploma_bucket(_df_split['diploma_type_id'])
    _df_split['diploma_type_bucket'] = pd.Categorical(
        _raw_mapped,
        categories=_bucket_categories,
    )
    print(f'  {_split_name:<6}: diploma_type_bucket assigned  '
          f'dtype={_df_split["diploma_type_bucket"].dtype}  '
          f'null_count={int(_df_split["diploma_type_bucket"].isna().sum())}')

print(f'\ndiploma_type_bucket categories: {list(df_train["diploma_type_bucket"].cat.categories)}')

  train : diploma_type_bucket assigned  dtype=category  null_count=0
  valid : diploma_type_bucket assigned  dtype=category  null_count=0
  test  : diploma_type_bucket assigned  dtype=category  null_count=0

diploma_type_bucket categories: [13, 15, 16, 19, 26, 6, -1]


**What this shows:** Confirms `diploma_type_bucket` was created in all three splits with
the correct `category` dtype. Null count should be 0 — all values (including original
nulls) are mapped to `-1` rather than left as `NaN`.

**Step B.6** — Full diagnostics: mapping summary, rare-code breakdown, unseen coverage
per split, raw vs bucket distributions side-by-side.

In [11]:
# ── Step B.6 Diagnostics ──

print('=' * 70)
print('D1 — Top-5 raw diploma codes and their bucket mapping')
print('=' * 70)
for _code in TOP_DIPLOMA_CODES:
    _cnt = int((df_train['diploma_type_id'] == _code).sum())
    print(f'  raw code {int(_code):>4}  ->  bucket {int(_code):>4}  (kept as own category)  '
          f'train count={_cnt:>8,}')


print()
print('=' * 70)
print(f'D2 — Rare-code bucket ({RARE_BUCKET_LABEL}): distinct train codes mapped to it')
print('=' * 70)
_rare_sorted = sorted(int(c) for c in _rare_codes)
print(f'  Distinct raw train codes in rare bucket : {len(_rare_sorted)}')
if _rare_sorted:
    print(f'  Codes : {_rare_sorted}')
    for _code in _rare_sorted:
        _cnt = int((df_train['diploma_type_id'] == _code).sum())
        print(f'    code {_code:>4}  train count={_cnt:>6,}')
else:
    print('  (No rare codes — all train codes are in top 5)')


print()
print('=' * 70)
print(f'D3 — Unseen-bucket ({UNSEEN_BUCKET_LABEL}) rows per split')
print('     (codes never in df_train, or null diploma_type_id)')
print('=' * 70)
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n_unseen = int((_df_split['diploma_type_bucket'] == UNSEEN_BUCKET_LABEL).sum())
    _n_total  = len(_df_split)
    _pct      = _n_unseen / _n_total * 100
    # Decompose: null original vs. genuinely unseen code
    _n_null_orig = int(_df_split['diploma_type_id'].isna().sum())
    _n_new_code  = _n_unseen - _n_null_orig
    print(f'  {_split_name:<6}: unseen-bucket rows = {_n_unseen:>7,}  ({_pct:.2f}% of split)')
    print(f'           of which: null diploma_type_id = {_n_null_orig:>7,}')
    print(f'                     unseen non-null code  = {_n_new_code:>7,}')


print()
print('=' * 70)
print('D4 — Raw diploma_type_id distribution per split')
print('=' * 70)
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    print(f'  {_split_name}:')
    _vc = _df_split['diploma_type_id'].value_counts(dropna=False).sort_values(ascending=False)
    for _idx, _cnt in _vc.items():
        print(f'    {str(_idx):>6} : {_cnt:>8,}')


print()
print('=' * 70)
print('D5 — diploma_type_bucket distribution per split (side-by-side with raw)')
print('=' * 70)

for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _raw_vc    = _df_split['diploma_type_id'].value_counts(dropna=False)
    _bucket_vc = _df_split['diploma_type_bucket'].value_counts(dropna=False).sort_index()
    print(f'\n  [{_split_name}] diploma_type_bucket distribution:')
    for _bval in _bucket_categories:
        _bcnt = int(_bucket_vc.get(_bval, 0))
        _pct  = _bcnt / len(_df_split) * 100
        _label = (
            f'top-5 code {_bval}'
            if _bval in _top5_set else
            f'rare bucket     ' if _bval == RARE_BUCKET_LABEL else
            f'unseen/null     '
        )
        print(f'    bucket {_bval:>4}  ({_label:18})  count={_bcnt:>8,}  ({_pct:.2f}%)')

D1 — Top-5 raw diploma codes and their bucket mapping
  raw code   15  ->  bucket   15  (kept as own category)  train count= 426,684
  raw code   16  ->  bucket   16  (kept as own category)  train count=  13,701
  raw code   13  ->  bucket   13  (kept as own category)  train count=   5,598
  raw code   19  ->  bucket   19  (kept as own category)  train count=   2,334
  raw code   26  ->  bucket   26  (kept as own category)  train count=     881

D2 — Rare-code bucket (6): distinct train codes mapped to it
  Distinct raw train codes in rare bucket : 9
  Codes : [9, 10, 14, 18, 20, 21, 22, 32, 69]
    code    9  train count=    43
    code   10  train count=   167
    code   14  train count=    72
    code   18  train count=   131
    code   20  train count=    83
    code   21  train count=   322
    code   22  train count=   101
    code   32  train count=   274
    code   69  train count=    43

D3 — Unseen-bucket (-1) rows per split
     (codes never in df_train, or null diploma_type

**What this shows:**
- **D1**: The 5 raw codes kept as individual categories and their training frequencies.
- **D2**: Which train codes fell into the rare bucket and how often they appeared.
- **D3**: How many rows per split ended up in the unseen (-1) bucket, split between
  genuine nulls from the diploma merge and codes not present in train.
- **D4**: Raw `diploma_type_id` distribution — use this as the reference before any bucketing.
- **D5**: Final `diploma_type_bucket` distribution, confirming bucketing collapsed correctly.

Review D3 carefully: valid/test unseen counts should be small if the diploma source covers
most students. Large unseen percentages in valid/test indicate codes the model has never
seen and will treat as cold-start.

**Step B.7** — Assert exactly ONE new column (`diploma_type_bucket`) was added to the
base column count. `diploma_type_id` and `diploma_gpa` already existed from
`merge_diploma.py` — they do not count as new here.

In [12]:
# ── Step B.7: Assert exactly 1 new column was added ──
# Uses BASE_COL_COUNT_* captured in Step A.1b (independent pre-bucketing
# snapshot) — NOT derived backward from the current (post-bucketing) shape.

NEW_BUCKET_COLS = ['diploma_type_bucket']

print(f'Base column count (captured in Step A.1b, before bucketing):')
print(f'  train : {BASE_COL_COUNT_TRAIN}')
print(f'  valid : {BASE_COL_COUNT_VALID}')
print(f'  test  : {BASE_COL_COUNT_TEST}')
print(f'\nColumn count after bucketing:')
print(f'  train : {df_train.shape[1]}')
print(f'  valid : {df_valid.shape[1]}')
print(f'  test  : {df_test.shape[1]}')

assert df_train.shape[1] == BASE_COL_COUNT_TRAIN + len(NEW_BUCKET_COLS), (
    f'FAIL (train): expected {BASE_COL_COUNT_TRAIN + len(NEW_BUCKET_COLS)} columns, '
    f'got {df_train.shape[1]}. Expected new column(s): {NEW_BUCKET_COLS}'
)
assert df_valid.shape[1] == BASE_COL_COUNT_VALID + len(NEW_BUCKET_COLS), (
    f'FAIL (valid): expected {BASE_COL_COUNT_VALID + len(NEW_BUCKET_COLS)} columns, '
    f'got {df_valid.shape[1]}. Expected new column(s): {NEW_BUCKET_COLS}'
)
assert df_test.shape[1] == BASE_COL_COUNT_TEST + len(NEW_BUCKET_COLS), (
    f'FAIL (test): expected {BASE_COL_COUNT_TEST + len(NEW_BUCKET_COLS)} columns, '
    f'got {df_test.shape[1]}. Expected new column(s): {NEW_BUCKET_COLS}'
)

# Verify the column is present in all three splits with the correct dtype
for _col in NEW_BUCKET_COLS:
    for _label, _df_e in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
        if _col not in _df_e.columns:
            raise AssertionError(f'Missing column "{_col}" in {_label} split.')
        if not hasattr(_df_e[_col], 'cat'):
            raise AssertionError(
                f'Column "{_col}" in {_label} split is not category dtype '
                f'(got {_df_e[_col].dtype}).'
            )

print(f'\nColumn count assertion PASSED (verified against independent pre-bucketing snapshot).')
print(f'New column verified in all splits with category dtype:')
for _col in NEW_BUCKET_COLS:
    print(f'  {_col}')


Base column count (captured in Step A.1b, before bucketing):
  train : 70
  valid : 70
  test  : 70

Column count after bucketing:
  train : 71
  valid : 71
  test  : 71

Column count assertion PASSED (verified against independent pre-bucketing snapshot).
New column verified in all splits with category dtype:
  diploma_type_bucket


**What this shows:** A hard guarantee that exactly one new column was added and that
it has the correct `category` dtype in all three splits. If the assertion fires, it
means either a column was added twice (re-run without the A.1b drop cell) or a column
was inadvertently created elsewhere in this notebook.

**Save** — Overwrite `SPLIT_DATA_DIR/df_train/valid/test.parquet` with the enriched
versions. The column count check is done above; this cell only writes to disk.

In [13]:
# ── Save: overwrite split parquet files with diploma_type_bucket added ──
#
# diploma_type_id is retained as an audit-only column (NOT in MODEL_FEATURES).
# diploma_type_bucket is the new model-facing feature — update MODEL_FEATURES
# and CATEGORICAL_FEATURES in model_training.py in a SEPARATE follow-up step
# after reviewing the diagnostics printed above.
#
# diploma_gpa was already present from merge_diploma.py and is NOT processed here.
# Add it directly to MODEL_FEATURES in the same follow-up step.

df_train.to_parquet(TRAIN_PATH, index=True)
df_valid.to_parquet(VALID_PATH, index=True)
df_test.to_parquet(TEST_PATH,   index=True)

print('Saved enriched splits:')
for _label, _path, _df_e in [
    ('train', TRAIN_PATH, df_train),
    ('valid', VALID_PATH, df_valid),
    ('test',  TEST_PATH,  df_test),
]:
    _size_mb = os.path.getsize(_path) / 1_048_576
    print(f'  {_label:<6}: {_df_e.shape}  ->  {_path}  ({_size_mb:.1f} MB)')

print()
print('diploma_type_bucket added to all splits.')
print('Next step: review the diagnostics above, then update MODEL_FEATURES and')
print('CATEGORICAL_FEATURES in model_training.py as a separate follow-up step.')

Saved enriched splits:
  train : (450465, 71)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_train.parquet  (18.8 MB)
  valid : (156097, 71)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_valid.parquet  (6.4 MB)
  test  : (110008, 71)  ->  D:/AI/Real projects/Academic_Advisor/data/model_data\df_test.parquet  (4.5 MB)

diploma_type_bucket added to all splits.
Next step: review the diagnostics above, then update MODEL_FEATURES and
CATEGORICAL_FEATURES in model_training.py as a separate follow-up step.


**What this shows:** File path, shape, and on-disk size confirm all three splits saved
with the new `diploma_type_bucket` column. The raw `diploma_type_id` and `diploma_gpa`
columns are preserved unchanged.  
**Do not update `MODEL_FEATURES` or `CATEGORICAL_FEATURES` until you have reviewed
the Step B.6 diagnostics and confirmed the bucket distribution is as expected.**